In [1]:
!pip install sentence-transformers faiss-cpu nltk flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.9 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import faiss
import nltk
from sentence_transformers import SentenceTransformer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [3]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
data = [
    {"question": "What room types are available?",
     "answer": "We offer Standard, Deluxe, Executive Suite, Family Suite."},

    {"question": "What is the price of standard room?",
     "answer": "Standard room costs 50 USD per night."},

    {"question": "What is the price of deluxe room?",
     "answer": "Deluxe room costs 80 USD per night."},

    {"question": "What amenities are available?",
     "answer": "We provide WiFi, AC, TV, Room Service, Breakfast."},

    {"question": "Is breakfast included?",
     "answer": "Yes, breakfast is free for all guests."},

    {"question": "How can I book a room?",
     "answer": "You can book through website or call reception."}
]

In [7]:
nltk.download('punkt_tab')
def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalnum()]
    tokens = [t for t in tokens if t not in stopwords.words('english')]
    return " ".join(tokens)

questions = [preprocess(item["question"]) for item in data]
answers = [item["answer"] for item in data]

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
question_vectors = model.encode(questions)

In [10]:
dimension = question_vectors.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(question_vectors))

In [11]:
def get_answer(query):
    query = preprocess(query)
    query_vec = model.encode([query])

    D, I = index.search(np.array(query_vec), k=1)

    return answers[I[0][0]]

In [12]:
print(get_answer("What rooms do you have?"))
print(get_answer("price of deluxe room"))
print(get_answer("Do you provide breakfast?"))

We offer Standard, Deluxe, Executive Suite, Family Suite.
Deluxe room costs 80 USD per night.
Yes, breakfast is free for all guests.
